<a href="https://colab.research.google.com/github/MoeTahech/real-fake-face-detection-thesis/blob/main/notebooks/10b_clip_vitl14_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# C — A fairer CLIP test: ViT-L/14 versus the ViT-B/32 of Phase 9

**Question.** Phase 9 used the small CLIP (ViT-B/32) and only its final output. Published work that generalizes well uses a larger CLIP (ViT-L/14). Does a **bigger frozen CLIP** detect fakes from generators it never saw?

**Design (same recipe as Phase 9, only the backbone changes).**
- CLIP is **frozen**. It only turns each image into numbers (features). Nothing in CLIP is trained.
- A **linear classifier** (one layer) is trained on those numbers: Adam, learning rate 1e-3, batch 256, 15 epochs, best validation epoch kept.
- **Training data:** (S) StyleGAN train split, 100,000 images, validation = StyleGAN validation split (20,000), exactly as in Phase 9. Optional (S+F): the same StyleGAN + face-swap sample as Phase 6 (60,000 train / 8,000 validation).
- **Test data (frozen suite, same subsets as notebook B):** StyleGAN test (5,000 + 5,000), face-swap test (5,000 + 5,000), DiffusionFace SD v1.5 and v2.1 with both real-image pairings (3,000 + 3,000 each). The old Stable Diffusion v1.4 set is **not used**.
- **Features compared:** the final projected embedding (`proj`, what Phase 9 used) and the CLS token of several intermediate layers (exploratory). **All layers are reported; do not pick the best one by looking at the test sets** (that would be selecting on the test data).
- A second backbone, **ViT-B/32 with the same pipeline**, is run too, so the comparison with L/14 is like for like.

**Runtime.** GPU (T4), roughly 40–70 minutes the first time, almost all of it feature extraction. Features are saved to Drive in shards, so you can stop and resume.

In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
import os
CHECKPOINT_DIR = '/content/drive/MyDrive/thesis_checkpoints'
OUT_DIR = f'{CHECKPOINT_DIR}/clip_vitl14'
os.makedirs(OUT_DIR, exist_ok=True)

# ---- settings ----
BACKBONES = {                      # tag: (huggingface name, intermediate layers whose CLS token is stored, batch size)
    'L14': ('openai/clip-vit-large-patch14', [12, 16, 20, 24], 128),
    'B32': ('openai/clip-vit-base-patch32',  [6, 9, 12],       256),
}
RUN_B32 = True            # like-for-like baseline (small, quick)
RUN_MIXED = True          # also train on StyleGAN + face-swap (Phase 6 sample)
TRAIN_PER_CLASS = None    # None = full 50,000 per class (as Phase 9). Use e.g. 15000 for a quick trial.
N_BOOT = 500              # bootstrap resamples for the confidence intervals
SHARD = 10000
print('Outputs go to:', OUT_DIR)

Wed Oct 21 11:26:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 1 — Download the data (token typed at runtime, not stored)

In [ ]:
from getpass import getpass
os.environ['KAGGLE_API_TOKEN'] = getpass('Paste your Kaggle API token (not saved in the notebook): ')
!pip install -q kaggle transformers
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces
!unzip -q -o 140k-real-and-fake-faces.zip -d data_raw
!kaggle datasets download -d manjilkarki/deepfake-and-real-images
!unzip -q -o deepfake-and-real-images.zip -d data_raw_2
print('Kaggle datasets ready.')

Paste your Kaggle API token (not saved in the notebook): ··········
Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
140k-real-and-fake-faces.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset URL: https://www.kaggle.com/datasets/manjilkarki/deepfake-and-real-images
License(s): unknown
deepfake-and-real-images.zip: Skipping, found more recently modified local copy (use --force to force download)
Kaggle datasets ready.


In [ ]:
import hashlib
WORK = '/content/diffface'; os.makedirs(WORK, exist_ok=True)
P1 = 'https://zenodo.org/records/10865300/files/{}?download=1'
P2 = 'https://zenodo.org/records/10865065/files/{}?download=1'
FILES = {
    'Real':     (P1.format('Real.tar'), 'Real.tar', '72379bcecf6382d7999cce59b33dfcdc'),
    'SD15_T2I': (P2.format('stable_diffusion_v_1_5_text2img_p3g7.tar'), 'stable_diffusion_v_1_5_text2img_p3g7.tar', '37a92f1973d6cc8941c8d98a195c09b4'),
    'SD21_T2I': (P2.format('stable_diffusion_v_2_1_text2img_p0g5.tar'), 'stable_diffusion_v_2_1_text2img_p0g5.tar', '55bd322166cfdb3bda9aa05cb5a545aa'),
}
def md5_of(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()
for key, (url, fname, md5) in FILES.items():
    path = f'{WORK}/{fname}'
    if not os.path.exists(path):
        print('Downloading', fname); get_ipython().system(f'wget -c -q --show-progress -O "{path}" "{url}"')
    print(f'{fname:<48} md5', 'OK' if md5_of(path) == md5 else 'MISMATCH - delete the file and re-run this cell')
    dest = f'{WORK}/{key}'
    if not os.path.isdir(dest) or not os.listdir(dest):
        os.makedirs(dest, exist_ok=True); print('Extracting', fname); get_ipython().system(f'tar -xf "{path}" -C "{dest}"')
print('DiffusionFace ready.')

Real.tar                                         md5 OK
stable_diffusion_v_1_5_text2img_p3g7.tar         md5 OK
stable_diffusion_v_2_1_text2img_p0g5.tar         md5 OK
DiffusionFace ready.


### Step 2 — Training lists and the frozen test sets (same functions and seeds as your earlier notebooks)

In [ ]:
import random, json, numpy as np
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
def find_real_fake_folders(root):
    out = []
    for dp, dn, _ in os.walk(root):
        low = [d.lower() for d in dn]
        if 'real' in low and 'fake' in low: out.append(dp)
    return out
def find_split(root, kw):
    m = [x for x in find_real_fake_folders(root) if kw in x.lower()]
    if not m: raise FileNotFoundError(f"No '{kw}' folder under {root}")
    return m[0]
def find_class_dir(split_root, cls):
    for d in os.listdir(split_root):
        if d.lower() == cls: return os.path.join(split_root, d)
    raise FileNotFoundError(cls)
def find_all_images(root, exts=IMG_EXTS):
    out = []
    for dp, _, fn in os.walk(root):
        for f in fn:
            if f.lower().endswith(exts): out.append(os.path.join(dp, f))
    return sorted(out)

styleGAN_train = find_split('data_raw', 'train'); styleGAN_val = find_split('data_raw', 'val'); styleGAN_test = find_split('data_raw', 'test')
faceswap_train = find_split('data_raw_2', 'train'); faceswap_test = find_split('data_raw_2', 'test')

def list_items(root, per_class=None, seed=0):
    items = []
    for cls in ['real', 'fake']:
        cd = find_class_dir(root, cls); files = sorted(os.listdir(cd))
        if per_class: random.Random(seed).shuffle(files); files = files[:per_class]
        items += [(os.path.join(cd, f), int(cls == 'real')) for f in files]
    return items

# (S) StyleGAN only, as in Phase 9
S_train = list_items(styleGAN_train, TRAIN_PER_CLASS)
S_val   = list_items(styleGAN_val, None if TRAIN_PER_CLASS is None else max(1000, TRAIN_PER_CLASS // 5))

# (S+F) StyleGAN + face-swap, same selection logic as build_p6_data (seed 42, 15,000 train + 2,000 val per class and source)
def p6_lists(n_train=15000, n_val=2000, seed=42):
    rng = random.Random(seed); tr, va = [], []
    for cls in ['real', 'fake']:
        for src_root in [styleGAN_train, faceswap_train]:
            cd = find_class_dir(src_root, cls); files = sorted(os.listdir(cd)); rng.shuffle(files)
            chosen = files[:n_train + n_val]
            tr += [(os.path.join(cd, f), int(cls == 'real')) for f in chosen[:n_train]]
            va += [(os.path.join(cd, f), int(cls == 'real')) for f in chosen[n_train:]]
    return tr, va
M_train, M_val = p6_lists() if RUN_MIXED else ([], [])

# frozen test sets (identical to notebook B, WITHOUT the old Stable Diffusion v1.4 set)
def subset_items(source_root, n_per_class, seed=99):
    rng = random.Random(seed); items = []
    for cls in ['real', 'fake']:
        cd = find_class_dir(source_root, cls); files = sorted(os.listdir(cd)); rng.shuffle(files)
        items += [(os.path.join(cd, f), int(cls == 'real')) for f in files[:n_per_class]]
    return items
EVAL = {'stylegan': subset_items(styleGAN_test, 5000), 'faceswap': subset_items(faceswap_test, 5000)}
SEED, N = 7, 3000
def sample_paths(key, n):
    paths = find_all_images(f'{WORK}/{key}'); random.Random(SEED).shuffle(paths); return paths[:n]
real_df = sample_paths('Real', N)
fake_df = {k: sample_paths(k, N) for k in ['SD15_T2I', 'SD21_T2I']}
ffhq_dir = find_class_dir(styleGAN_test, 'real')
ffhq_all = sorted(os.path.join(ffhq_dir, f) for f in os.listdir(ffhq_dir)); random.Random(SEED).shuffle(ffhq_all); ffhq_real = ffhq_all[:N]
for pair, reals in [('A', real_df), ('B', ffhq_real)]:
    for fk, short in [('SD15_T2I', 'SD15'), ('SD21_T2I', 'SD21')]:
        n = min(len(reals), len(fake_df[fk]))
        EVAL[f'DF_{pair}_{short}'] = [(p, 1) for p in reals[:n]] + [(p, 0) for p in fake_df[fk][:n]]

def uniq(*lists):
    seen, out = set(), []
    for L in lists:
        for p, _ in L:
            if p not in seen: seen.add(p); out.append(p)
    return out
ALL = uniq(S_train, S_val, M_train, M_val, *EVAL.values())
IDX = {p: i for i, p in enumerate(ALL)}
print('train S:', len(S_train), '| val S:', len(S_val), '| train S+F:', len(M_train), '| val S+F:', len(M_val))
for k, v in EVAL.items(): print(f'{k:<12} n={len(v):>6}  real={sum(y for _, y in v)}  fake={sum(1 - y for _, y in v)}')
print('unique images to embed:', len(ALL))

train S: 100000 | val S: 20000 | train S+F: 60000 | val S+F: 8000
stylegan     n= 10000  real=5000  fake=5000
faceswap     n= 10000  real=5000  fake=5000
DF_A_SD15    n=  6000  real=3000  fake=3000
DF_A_SD21    n=  6000  real=3000  fake=3000
DF_B_SD15    n=  6000  real=3000  fake=3000
DF_B_SD21    n=  6000  real=3000  fake=3000
unique images to embed: 184541


### Step 3 — Extract the frozen CLIP features (saved to Drive in shards; safe to stop and resume)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPModel
from PIL import Image
from tqdm.auto import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

clip_tf = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC), transforms.ToTensor(),
    transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])])
class PathDS(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return clip_tf(Image.open(self.paths[i]).convert('RGB'))

def extract_backbone(tag):
    hf_name, layers, bs = BACKBONES[tag]
    keys = ['proj'] + [f'L{l}' for l in layers]
    model = None; parts = {k: [] for k in keys}
    for s0 in range(0, len(ALL), SHARD):
        sp = ALL[s0:s0 + SHARD]; f = f'{OUT_DIR}/feat_{tag}_{s0}.pt'
        d = None
        if os.path.exists(f):
            d = torch.load(f)
            if not (d['n'] == len(sp) and d['first'] == sp[0] and d['last'] == sp[-1]): d = None
        if d is None:
            if model is None: model = CLIPModel.from_pretrained(hf_name).to(device).eval()
            loader = DataLoader(PathDS(sp), batch_size=bs, shuffle=False, num_workers=2)
            acc = {k: [] for k in keys}
            with torch.no_grad():
                for x in tqdm(loader, desc=f'{tag} shard {s0 // SHARD + 1}/{(len(ALL) + SHARD - 1) // SHARD}'):
                    x = x.to(device)
                    with torch.autocast('cuda', dtype=torch.float16, enabled=(device.type == 'cuda')):
                        out = model.vision_model(pixel_values=x, output_hidden_states=True)
                        acc['proj'].append(model.visual_projection(out.pooler_output).half().cpu())
                        for l in layers: acc[f'L{l}'].append(out.hidden_states[l][:, 0, :].half().cpu())
            d = {k: torch.cat(v) for k, v in acc.items()}; d.update(n=len(sp), first=sp[0], last=sp[-1])
            torch.save(d, f)
        for k in keys: parts[k].append(d[k])
    del model; torch.cuda.empty_cache()
    return {k: torch.cat(v) for k, v in parts.items()}

FEATS = {}
for tag in (['L14'] + (['B32'] if RUN_B32 else [])):
    FEATS[tag] = extract_backbone(tag)
    print(tag, {k: tuple(v.shape) for k, v in FEATS[tag].items()})

L14 {'proj': (184541, 768), 'L12': (184541, 1024), 'L16': (184541, 1024), 'L20': (184541, 1024), 'L24': (184541, 1024)}


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

B32 shard 9/19:   0%|          | 0/40 [00:00<?, ?it/s]

B32 shard 10/19:   0%|          | 0/40 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e83c1153a60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
Exception ignored in:   File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    <function _MultiProcessingDataLoaderIter.__del__ at 0x7e83c1153a60>assert self._parent_pid == os.getpid(), 'can only test a child process'

Traceback (most recent call last):
AssertionError  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
:     can only test a child processself._shutdown_workers()

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

...

B32 shard 17/19:   0%|          | 0/40 [00:00<?, ?it/s]

B32 shard 18/19:   0%|          | 0/40 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e83c1153a60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e83c1153a60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

B32 shard 19/19:   0%|          | 0/18 [00:00<?, ?it/s]

B32 {'proj': (184541, 512), 'L6': (184541, 768), 'L9': (184541, 768), 'L12': (184541, 768)}


### Step 4 — Train the linear classifiers

In [ ]:
def rows_of(items): return torch.tensor([IDX[p] for p, _ in items]), torch.tensor([y for _, y in items])

def train_head(Xtr, ytr, Xva, yva, epochs=15, lr=1e-3, bs=256, seed=0, standardize=True):
    torch.manual_seed(seed); g = torch.Generator().manual_seed(seed)
    Xtr, ytr, Xva, yva = Xtr.to(device), ytr.to(device), Xva.to(device), yva.to(device)
    if standardize:
        mu = Xtr.float().mean(0, keepdim=True); sd = Xtr.float().std(0, keepdim=True) + 1e-6
    else:
        mu = torch.zeros(1, Xtr.shape[1], device=device); sd = torch.ones(1, Xtr.shape[1], device=device)
    prep = lambda x: (x.float() - mu) / sd
    head = nn.Linear(Xtr.shape[1], 2).to(device); opt = torch.optim.Adam(head.parameters(), lr=lr)
    best, best_state = -1.0, None
    for ep in range(epochs):
        head.train(); perm = torch.randperm(len(Xtr), generator=g).to(device)
        for i in range(0, len(perm), bs):
            idx = perm[i:i + bs]
            loss = F.cross_entropy(head(prep(Xtr[idx])), ytr[idx]); opt.zero_grad(); loss.backward(); opt.step()
        head.eval()
        with torch.no_grad(): va = (head(prep(Xva)).argmax(1) == yva).float().mean().item()
        if va > best: best, best_state = va, {k: v.clone() for k, v in head.state_dict().items()}
    head.load_state_dict(best_state); head.eval()
    return head, prep, best

@torch.no_grad()
def p_real(head, prep, X): return torch.softmax(head(prep(X.to(device))), 1)[:, 1].cpu().numpy()

TRAIN_SETS = {'S': (S_train, S_val)}
if RUN_MIXED: TRAIN_SETS['S+F'] = (M_train, M_val)

PRED = {}      # (backbone, feature, train_set) -> {eval_set: (p_real, y)}
VAL_ACC = {}
for tag in FEATS:
    for feat, X in FEATS[tag].items():
        for tname, (trl, val) in TRAIN_SETS.items():
            itr, ytr = rows_of(trl); iva, yva = rows_of(val)
            head, prep, best = train_head(X[itr], ytr, X[iva], yva)
            VAL_ACC[(tag, feat, tname)] = best
            PRED[(tag, feat, tname)] = {s: (p_real(head, prep, X[rows_of(items)[0]]), np.array([y for _, y in items])) for s, items in EVAL.items()}
            print(f'{tag:<4} {feat:<5} train={tname:<4} best val acc {best:.4f}')

L14  proj  train=S    best val acc 0.9933
L14  proj  train=S+F  best val acc 0.9266
L14  L12   train=S    best val acc 0.9995
L14  L12   train=S+F  best val acc 0.9569
L14  L16   train=S    best val acc 0.9997
L14  L16   train=S+F  best val acc 0.9631
L14  L20   train=S    best val acc 0.9992
L14  L20   train=S+F  best val acc 0.9656
L14  L24   train=S    best val acc 0.9971
L14  L24   train=S+F  best val acc 0.9434
B32  proj  train=S    best val acc 0.9486
B32  proj  train=S+F  best val acc 0.8654
B32  L6    train=S    best val acc 0.9528
B32  L6    train=S+F  best val acc 0.8818
B32  L9    train=S    best val acc 0.9813
B32  L9    train=S+F  best val acc 0.9180
B32  L12   train=S    best val acc 0.9613
B32  L12   train=S+F  best val acc 0.8810


### Step 5 — Metrics and the results table

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score
def metrics(y, p, n_boot=N_BOOT, seed=0):
    pred = p >= 0.5
    real_recall = float(pred[y == 1].mean()); fake_recall = float((~pred[y == 0]).mean())
    lo = hi = float('nan')
    if n_boot:
        r = np.random.default_rng(seed); pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]; a = []
        for _ in range(n_boot):
            ii = np.concatenate([r.choice(pos, len(pos)), r.choice(neg, len(neg))]); a.append(roc_auc_score(y[ii], p[ii]))
        lo, hi = np.percentile(a, [2.5, 97.5])
    return dict(n=len(y), acc=float((pred == y).mean()), auc=float(roc_auc_score(y, p)), auc_lo=float(lo), auc_hi=float(hi),
                fake_recall=fake_recall, real_recall=real_recall)
rows = []
for (tag, feat, tname), d in PRED.items():
    for s, (p, y) in d.items():
        rows.append(dict(backbone=tag, feature=feat, train=tname, test_set=s, val_acc=VAL_ACC[(tag, feat, tname)], **metrics(y, p)))
long = pd.DataFrame(rows); long.to_csv(f'{OUT_DIR}/clip_results_long.csv', index=False)

SETS = ['stylegan', 'faceswap', 'DF_A_SD15', 'DF_A_SD21', 'DF_B_SD15', 'DF_B_SD21']
def table_for(train):
    out = {}
    for (tag, feat, tname) in PRED:
        if tname != train: continue
        r = {}
        for s in SETS:
            q = long[(long.backbone == tag) & (long.feature == feat) & (long.train == tname) & (long.test_set == s)].iloc[0]
            r[s] = f'{q.auc:.3f} [{q.auc_lo:.3f}, {q.auc_hi:.3f}]'
        out[f'{tag} {feat}'] = r
    return pd.DataFrame(out).T
tabS = table_for('S'); tabS.to_csv(f'{OUT_DIR}/table_train_S.csv')
print('AUC [95% CI], trained on StyleGAN only (S)'); tabS

AUC [95% CI], trained on StyleGAN only (S)


,stylegan,faceswap,DF_A_SD15,DF_A_SD21,DF_B_SD15,DF_B_SD21
L14 proj,"0.999 [0.999, 1.000]","0.534 [0.523, 0.545]","0.576 [0.561, 0.592]","0.570 [0.556, 0.584]","0.804 [0.793, 0.815]","0.789 [0.777, 0.799]"
L14 L12,"1.000 [1.000, 1.000]","0.565 [0.553, 0.577]","0.217 [0.206, 0.230]","0.142 [0.134, 0.150]","0.637 [0.624, 0.649]","0.519 [0.507, 0.530]"
L14 L16,"1.000 [1.000, 1.000]","0.578 [0.565, 0.588]","0.716 [0.703, 0.729]","0.593 [0.578, 0.606]","0.855 [0.846, 0.864]","0.760 [0.747, 0.771]"
L14 L20,"1.000 [1.000, 1.000]","0.525 [0.514, 0.537]","0.452 [0.439, 0.466]","0.393 [0.378, 0.407]","0.854 [0.844, 0.864]","0.817 [0.806, 0.828]"
L14 L24,"1.000 [1.000, 1.000]","0.562 [0.550, 0.573]","0.587 [0.572, 0.602]","0.612 [0.598, 0.627]","0.814 [0.803, 0.825]","0.823 [0.812, 0.835]"
B32 proj,"0.988 [0.986, 0.989]","0.506 [0.494, 0.518]","0.215 [0.204, 0.226]","0.285 [0.273, 0.298]","0.321 [0.307, 0.335]","0.396 [0.380, 0.410]"
B32 L6,"0.990 [0.989, 0.991]","0.542 [0.531, 0.553]","0.095 [0.088, 0.102]","0.108 [0.100, 0.117]","0.051 [0.046, 0.056]","0.079 [0.072, 0.087]"
B32 L9,"0.998 [0.998, 0.999]","0.548 [0.537, 0.560]","0.142 [0.133, 0.151]","0.198 [0.186, 0.208]","0.229 [0.218, 0.241]","0.305 [0.292, 0.319]"
B32 L12,"0.993 [0.992, 0.994]","0.522 [0.510, 0.532]","0.268 [0.256, 0.281]","0.315 [0.302, 0.329]","0.459 [0.445, 0.473]","0.495 [0.480, 0.509]"


In [ ]:
if RUN_MIXED:
    tabF = table_for('S+F'); tabF.to_csv(f'{OUT_DIR}/table_train_SF.csv')
    print('AUC [95% CI], trained on StyleGAN + face-swap (S+F)')
tabF if RUN_MIXED else None

AUC [95% CI], trained on StyleGAN + face-swap (S+F)


,stylegan,faceswap,DF_A_SD15,DF_A_SD21,DF_B_SD15,DF_B_SD21
L14 proj,"0.994 [0.993, 0.995]","0.856 [0.849, 0.863]","0.568 [0.553, 0.583]","0.632 [0.619, 0.648]","0.844 [0.834, 0.855]","0.869 [0.859, 0.878]"
L14 L12,"0.999 [0.999, 1.000]","0.787 [0.778, 0.796]","0.357 [0.343, 0.372]","0.322 [0.309, 0.334]","0.531 [0.517, 0.545]","0.483 [0.468, 0.497]"
L14 L16,"1.000 [0.999, 1.000]","0.836 [0.828, 0.844]","0.743 [0.731, 0.755]","0.564 [0.548, 0.576]","0.886 [0.877, 0.894]","0.744 [0.731, 0.755]"
L14 L20,"0.999 [0.999, 0.999]","0.894 [0.888, 0.901]","0.719 [0.705, 0.731]","0.534 [0.519, 0.549]","0.955 [0.950, 0.960]","0.906 [0.899, 0.913]"
L14 L24,"0.997 [0.996, 0.998]","0.888 [0.882, 0.895]","0.609 [0.595, 0.625]","0.644 [0.630, 0.659]","0.886 [0.877, 0.895]","0.895 [0.887, 0.903]"
B32 proj,"0.955 [0.952, 0.959]","0.816 [0.808, 0.824]","0.423 [0.410, 0.439]","0.524 [0.510, 0.539]","0.537 [0.521, 0.552]","0.629 [0.614, 0.643]"
B32 L6,"0.959 [0.955, 0.962]","0.767 [0.758, 0.775]","0.102 [0.094, 0.109]","0.141 [0.132, 0.150]","0.117 [0.108, 0.125]","0.164 [0.152, 0.174]"
B32 L9,"0.987 [0.985, 0.988]","0.842 [0.835, 0.850]","0.122 [0.114, 0.131]","0.155 [0.145, 0.165]","0.287 [0.276, 0.299]","0.328 [0.314, 0.342]"
B32 L12,"0.968 [0.964, 0.971]","0.841 [0.832, 0.849]","0.370 [0.357, 0.384]","0.521 [0.507, 0.536]","0.455 [0.441, 0.470]","0.610 [0.595, 0.625]"


In [ ]:
# Accuracy / fakes caught / reals kept for the main rows (final projected embedding)
main = long[(long.feature == 'proj')].copy()
for c in ['acc', 'auc', 'fake_recall', 'real_recall']: main[c] = main[c].round(3)
main[['backbone', 'train', 'test_set', 'n', 'auc', 'acc', 'fake_recall', 'real_recall']]

,backbone,train,test_set,n,auc,acc,fake_recall,real_recall
0,L14,S,stylegan,10000,0.999,0.992,0.990,0.993
1,L14,S,faceswap,10000,0.534,0.507,0.264,0.751
2,L14,S,DF_A_SD15,6000,0.576,0.566,0.266,0.866
3,L14,S,DF_A_SD21,6000,0.570,0.571,0.277,0.866
4,L14,S,DF_B_SD15,6000,0.804,0.630,0.266,0.994
5,L14,S,DF_B_SD21,6000,0.789,0.635,0.277,0.994
6,L14,S+F,stylegan,10000,0.994,0.964,0.965,0.964
7,L14,S+F,faceswap,10000,0.856,0.742,0.913,0.571
8,L14,S+F,DF_A_SD15,6000,0.568,0.537,0.400,0.674
9,L14,S+F,DF_A_SD21,6000,0.632,0.593,0.512,0.674


### Reference: the Phase 9 numbers you already have (ViT-B/32, trained on StyleGAN)
AUC on the frozen suite — StyleGAN 0.987, face-swap 0.486, DiffusionFace A SD1.5 0.303, A SD2.1 0.387, B SD1.5 0.463, B SD2.1 0.547.
The `B32 proj` row trained on S should be close to these (small differences are expected: this notebook standardizes the features and the head was retrained).

### What to send back
Paste the two AUC tables (S and S+F) and the `best val acc` lines from Step 4. In the thesis the **final-embedding row (`proj`) is the main comparison**; the intermediate layers are reported as exploratory, all of them, without picking a favourite.